# Least Squares and Ridge with Shu's final preprocessing

This notebook reviews the completed full-dataset experiment at Shu commit
`f6baef4`. It imports the reusable experiment, which calls the shared methods in
`implementations.py`. All transformations and L1 selection are fitted only on
training observations. The historical partial-feature notebook is preserved.

**Default:** display saved measured results. Set `RERUN = True` to rerun validation
and model selection (matching preprocessing checkpoints are reused). Full-data
candidate training/prediction remains in `run.py`, outside this notebook.

In [1]:
from pathlib import Path
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "implementations.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from experiments.evaluate_linear import evaluate
from experiments.linear_io import sha256

RERUN = False
results = evaluate() if RERUN else json.loads(
    (ROOT / "experiments/yusen_final_results.json").read_text())
print("Source:", results["source"]["shu_branch"]["commit"])
print("Environment:", results["environment"])
print("Split:", results["split"])


Source: f6baef4dd878ac79d030aa3c65cdda6e905a903b
Environment: {'python': '3.14.6', 'numpy': '2.4.6', 'MKL_NUM_THREADS': '1', 'OMP_NUM_THREADS': '1', 'selector_environment': {'python': '3.14.6', 'numpy': '2.4.6', 'MKL_NUM_THREADS': '4', 'OMP_NUM_THREADS': '4'}}
Split: {'seed': 42, 'stratified': False, 'train_rows': 246102, 'validation_rows': 82033, 'train_positives': 21737, 'validation_positives': 7238, 'train_ids_sha256': 'fe313b5f2644e33ef42b5f484fdb749018859af317ad1596f4bc299d6c578100', 'validation_ids_sha256': '8a5c2a22f886561283f4e196e70f75c866a544016bd0367276cd7f68b1295307'}


## Preprocessing and rank

The source uses 14 of 19 named decoding candidates; it does not use all 321 raw
variables. L1 settings match Shu: lambda 0.02, step 0.2, 2,000 iterations, cutoff
0.001. Least Squares requires independent columns, so rank is checked after L1
selection. Both linear models use the same retained columns. A separate Ridge
reference preserves the original selected representation.

In [2]:
p = results["preprocessing"]
print("Raw groups: one-value / ordered / decoded / categorical:",
      p["one_value"], p["ordered"], len(p["decoded"]), p["categorical"])
print("Unused decoding candidates:", p["decode_not_used"])
print("Expanded / L1-selected / independent columns:",
      p["expanded_columns"], p["l1_selected_columns"], results["rank"]["retained_columns"])
print("Removed after L1:", results["rank"]["dropped_names"])
print("Normalized condition number:", results["rank"]["normalized_condition_number"])


Raw groups: one-value / ordered / decoded / categorical: 5 111 14 157
Unused decoding candidates: ['HEIGHT3', 'EXERHMM1', 'EXERHMM2', 'FLSHTMY2', 'HIVTSTD3']
Expanded / L1-selected / independent columns: 1100 844 734
Removed after L1: ['CTELNUM1', '_PACAT1', '_PA150R2', 'HLTHPLN1=1', 'BPMEDS=nan', 'BLOODCHO=1', 'ASTHMA3=1', 'ASTHMA3=2', 'CHCSCNCR=2', 'HAVARTH3=1', 'HAVARTH3=2', 'ADDEPEV2=2', 'SEX=2', 'PREGNANT=nan', 'BLIND=2', 'DIFFWALK=2', 'SMOKE100=2', 'ARTHDIS2=2', 'HIVTST6=2', 'INSULIN=nan', 'DIABEDU=nan', 'CAREGIV1=nan', 'VIEYEXM2=9', 'VIGLUMA2=nan', 'ASATTACK=nan', 'STREHAB1=nan', 'CVDASPRN=1', 'ASPUNSAF=nan', 'RDUCHART=nan', 'RDUCSTRK=nan', 'ARTHWGT=nan', 'ARTHEXER=nan', 'HPVADVC2=nan', 'PROFEXAM=nan', 'BLDSTOOL=nan', 'HADSGCO1=nan', 'PCPSAAD2=nan', 'PCPSARE1=nan', 'PSATEST1=1', 'PSATEST1=nan', 'PCPSARS1=nan', 'CASTHDX2=nan', 'ADANXEV=nan', '_CPRACE=3', '_CPRACE=5', '_CPRACE=6', '_CPRACE=77', '_CPRACE=99', '_CPRACE=nan', '_RFHLTH=1', '_HCVU651=1', '_RFHYPE5=1', '_RFHYPE5=2', '_R

## Validation comparison

The earlier and current pipelines use the same 75/25 split, seed 42. Fixed results
use threshold -0.55 and Ridge lambda 0.001. Tuned results search six penalties and
91 thresholds on this validation set, so they are model-selection estimates,
not independent test scores. Labels are -1/+1; regression scores are not probabilities.

In [3]:
old = json.loads((ROOT / "experiments/yusen_baseline_results.json").read_text())
print(f"{'Model / protocol':43} {'Accuracy':>10} {'F1':>10} {'Precision':>10} {'Recall':>10}")
def show(label, values):
    print(f"{label:43} {values['accuracy']:10.6f} {values['f1']:10.6f} "
          f"{values['precision']:10.6f} {values['recall']:10.6f}")
for row in old["results"]:
    if row["model"] in ("Least Squares reduced", "Shared Ridge reduced"):
        show("Earlier " + row["model"], row)
show("Always negative", results["always_negative"])
for row in results["models"]:
    if row["method"] == "least_squares" or row["lambda"] == .001:
        show(row["method"] + " / final preprocessing, fixed", row["fixed_validation"])
for method, row in results["selected_models"].items():
    show(method + " / validation-selected", row["validation"])
    print("  lambda:", row["lambda"], "threshold:", row["threshold"])


Model / protocol                              Accuracy         F1  Precision     Recall
Earlier Shared Ridge reduced                  0.878098   0.343401   0.327202   0.361288
Earlier Least Squares reduced                 0.877123   0.344603   0.325473   0.366123
Always negative                               0.911767   0.000000   0.000000   0.000000
least_squares / final preprocessing, fixed    0.875782   0.428940   0.360833   0.528737
ridge / final preprocessing, fixed            0.875184   0.429614   0.359937   0.532744
least_squares / validation-selected           0.869260   0.429127   0.349035   0.556922
  lambda: None threshold: -0.57
ridge / validation-selected                   0.872783   0.429789   0.355477   0.543382
  lambda: 0.0001 threshold: -0.56


## Numerical diagnostics and overfitting checks

The recorded run verifies the loss formula, finite weights and the normal-equation
gradient. Training versus validation half-MSE is reported below; it does not by
itself prove generalization. F1 is independently recalculated from saved confusion
counts here. See `docs/verification.md` for the course and adapter tests.

In [4]:
for row in results["models"]:
    v = row["selected_validation"]
    assert v["tp"] + v["fp"] + v["fn"] + v["tn"] == results["split"]["validation_rows"]
    recalculated_f1 = 2 * v["tp"] / (2 * v["tp"] + v["fp"] + v["fn"])
    assert abs(recalculated_f1 - v["f1"]) < 1e-12
    print(row["method"], "lambda=", row["lambda"],
          "train half-MSE=", round(row["training_loss"], 6),
          "validation half-MSE=", round(v["half_mse"], 6),
          "normal-equation residual=", f"{row['max_normal_equation_residual']:.2e}")
print("Saved confusion counts and F1 are consistent.")


least_squares lambda= None train half-MSE= 0.129122 validation half-MSE= 0.130538 normal-equation residual= 6.12e-13
ridge lambda= 1e-05 train half-MSE= 0.129149 validation half-MSE= 0.130466 normal-equation residual= 4.96e-13
ridge lambda= 0.0001 train half-MSE= 0.129242 validation half-MSE= 0.130442 normal-equation residual= 4.98e-13
ridge lambda= 0.001 train half-MSE= 0.129991 validation half-MSE= 0.131026 normal-equation residual= 4.59e-13
ridge lambda= 0.01 train half-MSE= 0.13193 validation half-MSE= 0.132666 normal-equation residual= 3.88e-13
ridge lambda= 0.1 train half-MSE= 0.135622 validation half-MSE= 0.135962 normal-equation residual= 2.78e-13
ridge lambda= 1.0 train half-MSE= 0.142724 validation half-MSE= 0.142829 normal-equation residual= 2.45e-13
Saved confusion counts and F1 are consistent.


## Full-data candidate CSVs

Run `python run.py --model both` from the root to reproduce both candidate CSVs.
The included `models/yusen_linear_models.json` was fitted on all labeled rows;
use `--retrain` to refit weights. `--predict-only` requires compatible fitted state.
Generated data/caches are ignored by Git; the small fitted model is included. CSV format
checks are local; no AIcrowd test score is claimed.

In [5]:
manifest_path = ROOT / "experiments/yusen_submissions.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("Full-fit expanded / selected / model columns:", manifest["expanded_columns"],
          manifest["l1_selected_columns"], manifest["model_columns"])
    for method, record in manifest["outputs"].items():
        path = ROOT / record["path"]
        if path.exists():
            assert sha256(path) == record["sha256"]
        print(method, record["rows"], "rows; positives:", record["positive_predictions"])
        print("SHA-256:", record["sha256"])
else:
    print("Candidate CSV generation has not been recorded yet.")


Full-fit expanded / selected / model columns: 1104 838 731
least_squares 109379 rows; positives: 15394
SHA-256: bb2b8f894af772f591fa7741e6dac0054ee36d6afa2621969ced790b5c39f310
ridge 109379 rows; positives: 14799
SHA-256: e3a4f2c2a4d204e782fbbb9a2188e008224730670e71f1c97cb6e545145ba028


## Report and integration

- `docs/yusen_final_results.md`: concise measured results and limitations.
- `report/yusen_results.tex`: current Overleaf excerpt.
- `docs/integration.md`: source commits, ownership and merge guidance.
- `experiments/yusen_final_results.json`: exact metrics and all search results.

The new versus old pipeline comparison combines several preprocessing changes.
It cannot identify the isolated effect of decoding. Validation-selected results
may be optimistic. A final team model and its AIcrowd score still require team
confirmation; this notebook does not replace that decision.